In [ ]:
import requests
from datetime import datetime, timedelta
from pyspark.sql import Row
from pyspark.sql.functions import col, to_date, to_timestamp
from pyspark.sql.utils import AnalysisException
from pyspark.sql.functions import max as _max

In [ ]:
# SGS series codes for official PTAX daily close rates (buy/sell are separate series)
SGS_SERIES = {
    "USD": {"buy": 10813, "sell": 1},
    "EUR": {"buy": 21620, "sell": 21619},
}

In [ ]:
def fetch_bcb_data(currency, start_date, end_date):
    """Fetches buy and sell currency exchange rates from the Central Bank of Brazil (SGS API).

    The function splits the date range into annual intervals to prevent overly
    large requests, fetches the buy and sell time series mapped in the global
    `SGS_SERIES` dictionary, and consolidates the results.

    Args:
        currency (str): Currency identifier (e.g., 'USD', 'EUR') key matching
            an entry in `SGS_SERIES`.
        start_date (datetime): Start date for the exchange rate lookup.
        end_date (datetime): End date for the exchange rate lookup.

    Returns:
        list[Row]: A list of `Row` objects containing currency, buy rate,
        sell rate, and a formatted quote timestamp ('YYYY-MM-DD 13:00:00.000').

    Raises:
        ValueError: If `currency` is not mapped in the `SGS_SERIES` dictionary.
        requests.HTTPError: If an HTTP request fails for any reason other than a 404 status.
    """
    codes = SGS_SERIES.get(currency)
    if not codes:
        raise ValueError(f"No SGS series mapped for currency {currency}")

    all_rows = []
    current_start = start_date

    while current_start <= end_date:
        current_year_end = datetime(current_start.year, 12, 31)
        current_end = min(current_year_end, end_date)

        params_common = {
            "formato": "json",
            "dataInicial": current_start.strftime("%d/%m/%Y"),
            "dataFinal": current_end.strftime("%d/%m/%Y"),
        }

        def get_series(code):
            url = f"https://api.bcb.gov.br/dados/serie/bcdata.sgs.{code}/dados"
            resp = requests.get(url, params=params_common, timeout=15)
            if resp.status_code == 404:
                # SGS returns 404 (instead of an empty list) when there's no
                # published data for the requested range -- e.g. weekends,
                # holidays, or today's rate not published yet. Treat as "no data".
                return {}
            resp.raise_for_status()
            data_json = resp.json()
            
            if not isinstance(data_json, list):
                return {}

            return {item["data"]: float(str(item["valor"]).replace(",", ".")) for item in data_json}

        print(f"Requesting {currency} quotes from SGS ({params_common['dataInicial']} to {params_common['dataFinal']})...")
        buy_by_date = get_series(codes["buy"])
        sell_by_date = get_series(codes["sell"])

        for date_str, buy_val in buy_by_date.items():
            sell_val = sell_by_date.get(date_str)
            if sell_val is None:
                # Skip dates where one of the two series is missing a value
                continue
            dt = datetime.strptime(date_str, "%d/%m/%Y")
            all_rows.append(Row(
                currency=currency,
                buy_rate=buy_val,
                sell_rate=sell_val,
                # Placeholder time (SGS has no intraday timestamp); date portion is what matters downstream
                quote_date_time=dt.strftime("%Y-%m-%d 13:00:00.000")
            ))

        current_start = current_end + timedelta(days=1)

    return all_rows

In [ ]:
# CONFIGURATION
catalog_name = dbutils.widgets.get("catalog_name")
schema = dbutils.widgets.get("schema")
table_name_usd = f"{catalog_name}.{schema}.currency_rates_usd"
table_name_eur = f"{catalog_name}.{schema}.currency_rates_eur"
today = datetime.now()

In [ ]:
def get_start_date(table_name):
    """Determines the start date for data ingestion based on existing Delta table records.

    Queries the specified table for the maximum recorded quote date. If records
    exist, the function sets the start date to the day immediately following the
    latest record to enable an incremental load. If the table is empty or cannot
    be accessed, it falls back to a full historical load starting from 2020-01-01.

    Args:
        table_name (str): Name of the target PySpark/Delta table to query.

    Returns:
        datetime: The start date to be used for data extraction.
    """
    try:
        max_date_df = spark.table(table_name).select(_max("quote_date"))
        last_processed_date = max_date_df.collect()[0][0]

        if last_processed_date:
            start = datetime.combine(last_processed_date, datetime.min.time()) + timedelta(days=1)
            print(f"[{table_name}] Target Delta table found. Executing incremental load starting from: {start.strftime('%Y-%m-%d')}")
            return start
        else:
            raise AnalysisException("Table exists but contains no records.")

    except Exception as e:
        start = datetime(2020, 1, 1)
        print(f"[{table_name}] Initial table check skipped/failed ({type(e).__name__}). Initiating FULL HISTORICAL LOAD starting from: {start.strftime('%Y-%m-%d')}")
        return start

In [ ]:
def ingest_currency(currency, table_name):
    """Orchestrates incremental ingestion of currency exchange rates into a Delta table.

    The pipeline calculates the target `start_date` based on existing records, 
    fetches new data from the Central Bank of Brazil (SGS API) up to the current date,
    transforms the payload into a structured PySpark DataFrame, and appends the 
    records to the target Delta table.

    Args:
        currency (str): Currency identifier (e.g., 'USD', 'EUR') mapped in `SGS_SERIES`.
        table_name (str): Target PySpark/Delta table name where records are appended.
    """
    start_date = get_start_date(table_name)

    if start_date.date() > today.date():
        print(f"[{table_name}] Already up to date with today's date. No action required.")
        return

    print(f"[{table_name}] Requesting {currency} quotes from {start_date.strftime('%Y-%m-%d')} to {today.strftime('%Y-%m-%d')}...")
    data = fetch_bcb_data(currency, start_date, today)

    if not data:
        print(f"[{table_name}] The Central Bank API hasn't published new rates for this timeframe yet (weekends, holidays, or early daily execution).")
        return

    raw_df = spark.createDataFrame(data)

    # Parse the string timestamp and extract a pure date column for optimal querying
    incremental_df = raw_df \
        .withColumn("quote_timestamp", to_timestamp(col("quote_date_time"), "yyyy-MM-dd HH:mm:ss.SSS")) \
        .withColumn("quote_date", to_date(col("quote_timestamp"))) \
        .drop("quote_date_time") \
        .select("quote_date", "currency", "buy_rate", "sell_rate", "quote_timestamp")

    print(f"[{table_name}] Appending new records into Delta table...")
    incremental_df.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(table_name)

    print(f"[{table_name}] Incremental ingestion completed successfully! {incremental_df.count()} rows added.")

In [ ]:
# Run ingestion independently for USD and EUR, each into its own Delta table
ingest_currency("USD", table_name_usd)
ingest_currency("EUR", table_name_eur)